# WeldFatigueChecker · CoreML 救急导出（从 last.pt，5 分钟，无需训练）

## 为什么需要这个（2026-10-04 定位的真凶，已二次核实）
App 端 CoreML 模型**从第一天起就没工作过**，但原因**不是**“误用了 COCO 80 类权重”：
- `last.pt` 是真正的 5 类焊缝权重（本机已核实：类别数 5、Porosity 等五类名、PyTorch 输出 `[1,9,8400]`=4+5）。
- **真凶是 ultralytics 的 CoreML(MLProgram) 导出 bug #22309**：`IOSDetectModel.forward` 在 `mlprogram=True`（默认）时，把 5 类的类维度**强制 padding 到 80 通道**（`cls = F.pad(cls,(0,75))`），且 NMS 标签被垫成 80 个（前 5 个是焊缝名、后 75 个是 `5..79` 占位数字）。
- 部署的 `WeldDefectModel.mlpackage` 的 confidence 输出因此是 `[8400,80]`，而 App 解析端守卫要求 `nc == 5`（见 MLDefectDetector.swift:382）→ 80≠5 → 永远返回空 → 任何图、任何灵敏度档、任何域都 0 检出。

**本机已验证 `last.pt` 本身完全正常**：源1 抽样 Porosity mAP50=0.942，predict 置信度 0.62~0.77。只需重新导出**真正 5 通道**的 CoreML 即可救活 App。

## 上传到实例
- `last.pt`（120 权重，本机 `Desktop/log/last.pt`）→ 实例根目录

## 死穴（三令五申）
- 导出对象**只能是 last.pt（或训练出的 best.pt）**，绝不能是 yolov8n.pt！
- **必须用 monkeypatch 强制 `mlprogram=False`**：否则 ultralytics 会把类维度 pad 到 80，App 依旧全盲。
- 先不加 quantize（排除变量，fp16 约 6MB），App 能检出后再考虑量化。

In [ ]:
!pip install -q ultralytics coremltools
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)

In [ ]:
# ===== 第 0 步：清除本内核此前的补丁残留（关键！）=====
# 此前若在同一内核跑过带 forward 替换的补丁，旧补丁仍挂在类对象上，
# jit.trace 会无限递归（RecursionError）。reload 恢复原始类，再同步 exporter 引用。
import importlib
import ultralytics.utils.export.coreml as cm
import ultralytics.engine.exporter as ex
importlib.reload(cm)                      # 恢复原始 forward/__init__
ex.IOSDetectModel = cm.IOSDetectModel     # exporter 无论顶层引用还是调用时导入，都拿到同一个（干净）类
print('✅ 已清除内核补丁残留（原始 IOSDetectModel 已恢复）')

# 关键修正：禁用 ultralytics 把 5 类 padding 到 80 通道的 hack（issue #22309）
# 只改 __init__ 强制 mlprogram=False：原始 forward 里 `if self.mlprogram and ...` 因此不触发 → 类别不再 pad 到 80。
# ⚠️ 不要替换 forward 本身：torch 2.13 的 jit.trace 对“被替换的 forward 函数”会无限递归（RecursionError），原始 forward 没问题。
import ultralytics.utils.export.coreml as cm
_orig_ios_init = cm.IOSDetectModel.__init__
def _ios_init_no_pad(self, model, im, mlprogram=True):
    _orig_ios_init(self, model, im, mlprogram=False)
cm.IOSDetectModel.__init__ = _ios_init_no_pad
print('✅ 已应用 monkeypatch（__init__ 强制 mlprogram=False）')

from ultralytics import YOLO
import os
assert os.path.isfile('last.pt'), '未找到 last.pt，请先上传到实例根目录'
m = YOLO('last.pt')   # ⚠️ 只能用训练好的权重；YOLO('yolov8n.pt') = 导出错误权重
# mlmodel 保持默认 False → 产出 .mlpackage（MLProgram 现代格式，App 兼容）
out = m.export(format='coreml', nms=True, imgsz=640, conf=0.05)   # 先不量化，排除变量
print('导出产物:', out)
# Ultralytics 按权重文件名命名产物 → 重命名为工程所需名
if os.path.isdir('last.mlpackage'):
    if os.path.isdir('WeldDefectModel.mlpackage'):
        import shutil; shutil.rmtree('WeldDefectModel.mlpackage')
    os.rename('last.mlpackage', 'WeldDefectModel.mlpackage')
    print('已重命名 -> WeldDefectModel.mlpackage')


In [ ]:
# 硬验证：必须是真·5类（Linux 可跑，纯解析 spec，不推理）
# 决定性证据 = NMS 的 stringClassLabels：恰好 5 个焊缝类 = 修好；若是 80 个（前5焊缝名+后75数字） = bug未修。
# 注意：带 NMS 的 pipeline 模型里，confidence/coordinates 输出在 spec 中声明为空形状 []，
#       不能用其末维判定类别数（会取到 None 误报），故以 NMS 标签数组为准。
import coremltools as ct
import os, shutil
spec = ct.utils.load_spec('WeldDefectModel.mlpackage')
print('模型顶层类型:', spec.WhichOneof('Type'))
assert spec.WhichOneof('Type') == 'pipeline', '结构异常：应为 pipeline'

EXPECT = ['porosity', 'crack', 'undercut', 'overlap', 'unfused']
labels = None
for m in spec.pipeline.models:
    if m.HasField('nonMaximumSuppression'):
        labels = list(m.nonMaximumSuppression.stringClassLabels.vector)
        break
print('NMS 标签:', labels)
assert labels is not None, '❌ 未找到 NMS 节点'
assert labels == EXPECT, f'❌ 类别标签错误（应恰好 5 个焊缝类）: 得到={labels}'
assert len(labels) == 5, f'❌ 标签数={len(labels)}（应为5，80=bug未修）'
print('→ NMS 标签恰好为 5 个焊缝类，80 通道 padding bug 已消除')

# 设 NMS 内嵌 confidenceThreshold=0.05（App 端阈值逻辑依赖它）。
# Linux 重存可能失败 → 先备份原包，失败则还原（App 推理时也会自行传入该值，不影响可用性）。
bak = 'WeldDefectModel.mlpackage._verify_bak'
if os.path.exists(bak): shutil.rmtree(bak)
shutil.copytree('WeldDefectModel.mlpackage', bak)
try:
    for m in spec.pipeline.models:
        if m.HasField('nonMaximumSuppression'):
            m.nonMaximumSuppression.confidenceThreshold = 0.05
    ct.models.MLModel(spec, skip_model_load=True).save('WeldDefectModel.mlpackage')
    print('NMS confidenceThreshold 已设 0.05 并重存')
except Exception as e:
    print(f'⚠️ 阈值重存失败，已恢复原包（App 推理时会自行传入 confidenceThreshold）: {e}')
    if os.path.exists('WeldDefectModel.mlpackage'): shutil.rmtree('WeldDefectModel.mlpackage')
    shutil.copytree(bak, 'WeldDefectModel.mlpackage')
finally:
    if os.path.exists(bak): shutil.rmtree(bak)

print('✅ 验证通过：真·5 类焊缝模型')
print('可以下载 WeldDefectModel.mlpackage 了（左侧文件面板右键下载）')


## 拿回仓库
1. 下载 `WeldDefectModel.mlpackage`，覆盖仓库 `app_ios/native/WeldFatigueChecker/WeldDefectModel.mlpackage`。
2. 本机 Git Bash：
```
cd /d/workbuddy/workbuddy学习/weld_fatigue_checker
git add app_ios/native/WeldFatigueChecker/WeldDefectModel.mlpackage
git commit -m "fix(model): 重导 5 通道 CoreML，修复 MLProgram 导出 padding 到 80 导致 App 全盲"
git push origin main
```
3. 等 Actions 变绿 → 下载 IPA → 侧载 → 底栏核对 BUILD sha → 复测源1 图（应能检出）。